# What to learn: Memory

1. Basics: LLMs are stateless
   - Pass history manually as a list of messages
   - MessagesPlaceholder in ChatPromptTemplate

2. Short-term memory (within one conversation)
   - LangGraph checkpointer (InMemorySaver) + thread_id
   - Managing long histories: trimming messages, summarizing old ones

3. Conversational RAG
   - Rewrite a follow-up question using history before retrieval
     ("what about his Test record?" → "Virat Kohli's Test record")

4. Long-term memory (across conversations)
   - LangGraph Store: saving facts about a user, retrieving them later
   - Semantic search over stored memories

Legacy (skip): ConversationBufferMemory, ConversationSummaryMemory,
ConversationChain. Know what RunnableWithMessageHistory is, but don't focus on it.



In [ ]:

import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


#get llm

In [ ]:
from langchain_groq import ChatGroq

llm_gr = ChatGroq(model="openai/gpt-oss-20b")

# 1. Memory using MessagesPlaceholder - manual memory management

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

messages = [
    ("human","hi, how are you doing"),
    ("ai","Doing good, how can I help you today?"),
    ("human","Is Virat kohli a sports man?")
]

prompt = ChatPromptTemplate.from_messages(messages=messages)

chain = prompt | llm_gr
result = chain.invoke({})
messages.append(("ai", result.content))
print(result.content)

messages_next =[
    ("system","Designed as ai assistant"),
    MessagesPlaceholder(variable_name= "history"),
    ("human","what is his wife name?")
]

prompt = ChatPromptTemplate.from_messages(messages=messages_next)

chain = prompt | llm_gr
result = chain.invoke({"history":messages})
print(result.content)

# RunnableWithMessageHistor

1. Get prompt ready with messagepalceholder and prepare chain with llm
2. Prepare history with session id
3. Get chat - RunnableWithMessageHistory


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

#1. Get prompt ready with messagepalceholder and prepare chain with llm
messages =[
    ("system","Designed as ai assistant"),
    MessagesPlaceholder(variable_name= "history"),
    ("human","{question}")
]

#2. Prepare history with session id
prompt = ChatPromptTemplate(messages=messages)
chain = prompt | llm_gr


store = {}
def get_sessionId(sessionId):
    if sessionId not in store:
        store[sessionId] = InMemoryChatMessageHistory()
    return store[sessionId]

# 3. Get chat - RunnableWithMessageHistory
chat = RunnableWithMessageHistory(runnable=chain,get_session_history=get_sessionId,history_messages_key="history",input_messages_key="question")

cfg = {"configurable":{"session_id":"Vikram"}}

# answer = chat.invoke({"question":"What is name of Karnataka's chief minister?"},config=cfg)
# print(answer.content)

questions_list =[
    "What is name of Karnataka's chief minister?",
    "Does this minister has full power?",
    "Who was the CM before him?"
]

for question in questions_list:
    print("#####&&&&&&&&&&&&&&&&&&&&&&&&",question)
    answer = chat.invoke({"question":question},config=cfg)
    print(answer.content)


